# الفصل الثامن عشر: إدارة المشاريع كمحترف

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-18.ipynb)

> **الوقت المتوقع:** ~16 دقيقة قراءة · **10** قسمًا · **7** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **18.1** لماذا يستحق هذا الفصل اهتمامك الكامل؟
- **18.2** الجزء الأول: إدارة البيئات والتبعيات مع uv
- **18.3** الجزء الثاني: هيكلة مشروع احترافية
- **18.4** الجزء الثالث: أساسيات Git
- **18.5** أفضل الممارسات الشاملة
- **18.6** مشروع تطبيقي: تجهيز مشروع كامل من الصفر
- **18.7** تمارين
- **18.8** اختبر فهمك (Quiz)
- **18.9** ملخص الفصل (Cheat Sheet)
- **18.10** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

## 18.1 لماذا يستحق هذا الفصل اهتمامك الكامل؟

تخيّل أنك انتهيت للتو من بناء مشروع رائع بـ Python، وتريد مشاركته مع صديق ليجربه. ترسل له ملف `main.py`... فيحاول تشغيله، فيظهر له:

```
ModuleNotFoundError: No module named 'requests'
```

يُثبّت `requests`، يُشغّل البرنامج مجددًا، يظهر خطأ آخر من مكتبة مختلفة! بعد نصف ساعة من التثبيت اليدوي المتكرر، يكتشف أن إصدار مكتبة لديه مختلف عن إصدارك، والكود يتصرف بشكل مختلف تمامًا!

هذا بالضبط ما تحله **إدارة المشاريع الاحترافية**. وتخيّل سيناريو آخر: تعمل على ميزة جديدة في مشروعك، تُفسد شيئًا عن طريق الخطأ، وتتمنى لو تستطيع «الرجوع بالزمن» إلى النسخة التي كانت تعمل بشكل صحيح قبل ساعة. هذا بالضبط ما يحله **Git**.

هذا الفصل مختلف قليلًا عن سابقيه: لن نتعلم مفاهيم برمجية جديدة في اللغة نفسها، بل سنتعلم **الأدوات والممارسات** التي تفصل بين «شخص يكتب سكريبتات Python للمتعة» و«مبرمج محترف يبني برامج قابلة للمشاركة، والصيانة، والتطوير الجماعي». هذه المهارات ضرورية ومُسلَّم بها في كل بيئة عمل حقيقية، وستحتاجها بشدة في كل الفصول القادمة من الكتاب، خاصة عند بناء مشاريع الويب وواجهات API.

> 💡 **نصيحة الخبير:** كثير من المبرمجين المبتدئين يتجاهلون هذا الجانب لأنه «لا يبدو كبرمجة فعلية»، لكن الحقيقة أن إتقان إدارة المشاريع وGit يوفر عليك ساعات (وربما أيامًا) من الإحباط لاحقًا، ويجعلك قادرًا على العمل ضمن فريق بثقة تامة.

## 18.2 الجزء الأول: إدارة البيئات والتبعيات مع uv

### لماذا نحتاج Virtual Environments؟ (مراجعة وتعميق)

تذكر من الفصل الثاني أننا تحدثنا بإيجاز عن **البيئات الافتراضية (Virtual Environments)**. الآن حان وقت فهمها بعمق حقيقي.

### المشكلة: التعارض بين المشاريع

تخيّل أن لديك مشروعين على حاسوبك:
- **المشروع أ**: يحتاج `pandas` الإصدار 1.5 (لأن كودًا قديمًا فيه يعتمد على سلوك معيّن تغيّر لاحقًا).
- **المشروع ب**: يحتاج `pandas` الإصدار 2.2 (لاستخدام ميزة جديدة).

إن ثبّتَّ `pandas` «عالميًا» على حاسوبك (Globally)، فلا يمكنك أن يكون لديك إصدارين مختلفين في نفس الوقت! تثبيت الإصدار الجديد سيُفسد المشروع الأول، والعكس صحيح.

**الحل**: لكل مشروع **بيئته الخاصة المعزولة تمامًا**، بمكتباتها وإصداراتها الخاصة، دون أي تداخل مع مشاريع أخرى على نفس الجهاز.

```
حاسوبك
├── Python (التثبيت العام - لا نستخدمه مباشرة للمشاريع!)
├── المشروع أ/
│   └── .venv/  ← بيئة معزولة خاصة بهذا المشروع فقط (pandas 1.5)
└── المشروع ب/
    └── .venv/  ← بيئة معزولة خاصة بهذا المشروع فقط (pandas 2.2)
```

> ⚠️ **خطأ شائع جدًا عند المبتدئين:** تثبيت كل المكتبات مباشرة على النظام العام (`pip install` بدون بيئة افتراضية). هذا يؤدي حتمًا، عاجلًا أو آجلًا، إلى تعارضات بين المشاريع، ويجعل مشاركة مشروعك مع الآخرين أمرًا شبه مستحيل (لأنك لا تعرف بالضبط ما الذي يحتاجه مشروعك فعليًا من بين عشرات المكتبات المثبتة عالميًا على جهازك فقط).

### لماذا `uv` تحديدًا؟

اعتمدنا `uv` منذ الفصل الثاني كأداة إدارة الحزم والمشاريع الأساسية في هذا الكتاب. الآن دعنا نفهم بعمق لماذا هي الخيار الأمثل حاليًا، ونستكشف كل إمكانياتها.

### 🔄 مقارنة: تطور أدوات إدارة الحزم في Python

| الأداة | الوظيفة | الحالة |
|---|---|---|
| `pip` | تثبيت الحزم الأساسي | قديمة، لا تزال موجودة، بطيئة نسبيًا |
| `venv` | إنشاء بيئات افتراضية | جزء من المكتبة القياسية، بسيطة لكن محدودة |
| `virtualenv` | بديل أقدم وأكثر مرونة لـ `venv` | أقل استخدامًا الآن |
| `pipenv` | يجمع pip + virtualenv | انخفض استخدامها كثيرًا |
| `poetry` | إدارة شاملة للمشاريع والتبعيات | لا تزال شائعة، لكن أبطأ من uv |
| **`uv`** | **كل ما سبق، لكن أسرع بعشرات المرات، مكتوبة بلغة Rust** | **الخيار الموصى به حاليًا (2024-2025+)** |

`uv` طوّرتها شركة **Astral** (نفس الشركة التي طوّرت `ruff` الذي سنستخدمه في الفصل 24)، وهي تجمع في أداة واحدة: إدارة البيئات الافتراضية، تثبيت الحزم، إدارة إصدارات Python نفسها، وتشغيل السكريبتات — بسرعة تفوق الأدوات التقليدية بعشرات المرات بسبب كتابتها بلغة Rust عالية الأداء.

### إنشاء مشروع جديد باحترافية مع uv

حتى الآن في الكتاب، كنا ننشئ ملفات `.py` منفردة. الآن سنتعلم إنشاء **مشروع متكامل** بالهيكلية القياسية الصحيحة.

```bash
uv init my_project
cd my_project
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

هذا الأمر يُنشئ هيكلية كاملة:

```
my_project/
├── .venv/              # البيئة الافتراضية (تُنشأ لاحقًا عند أول uv run أو uv sync)
├── .gitignore          # ملف يُخبر Git بالملفات التي يجب تجاهلها
├── .python-version     # يُحدد إصدار Python المطلوب لهذا المشروع تحديدًا
├── pyproject.toml      # ملف التهيئة المركزي للمشروع (الأهم!)
├── README.md           # توثيق المشروع
└── main.py             # نقطة الدخول الأولية
```

> 💡 **نصيحة الخبير:** لاحظ ملف `.python-version`. هذا الملف يضمن أن أي شخص (بما فيك أنت بعد أشهر) يعمل على هذا المشروع سيستخدم **بالضبط** نفس إصدار Python المحدد، حتى لو كانت لديه إصدارات أخرى مثبتة على جهازه. `uv` تقرأ هذا الملف تلقائيًا وتُنزّل الإصدار المطلوب إن لم يكن موجودًا أصلًا!

### محتوى `pyproject.toml` الأساسي

```toml
[project]
name = "my-project"
version = "0.1.0"
description = "وصف موجز للمشروع"
readme = "README.md"
requires-python = ">=3.14"
dependencies = []

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

هذا الملف هو **«بطاقة هوية» المشروع بالكامل**: اسمه، إصدار Python المطلوب، وكل التبعيات (المكتبات الخارجية) التي يحتاجها.

### إدارة التبعيات (Dependencies)

### إضافة مكتبة جديدة

```bash
uv add requests
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

هذا الأمر يقوم بعدة أشياء تلقائيًا:
1. يُنشئ `.venv/` إن لم تكن موجودة.
2. يُثبّت `requests` وكل تبعياتها داخل هذه البيئة المعزولة.
3. يُضيف `requests` إلى قسم `dependencies` في `pyproject.toml`.
4. يُحدّث (أو يُنشئ) ملف `uv.lock`.

```toml
[project]
dependencies = [
    "requests>=2.31.0",
]
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

### إضافة مكتبة بإصدار محدد

```bash
uv add "requests==2.31.0"    # إصدار محدد بالضبط
uv add "pandas>=2.0,<3.0"    # نطاق إصدارات
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### حذف مكتبة

```bash
uv remove requests
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### تبعيات التطوير فقط: `--dev`

تذكر من الفصل 15 كيف ثبّتنا `mypy`:

```bash
uv add --dev mypy pytest ruff
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```toml
[dependency-groups]
dev = [
    "mypy>=1.8.0",
    "pytest>=8.0.0",
    "ruff>=0.3.0",
]
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

> 💡 **نصيحة الخبير - لماذا نفصل تبعيات التطوير؟** أدوات مثل `mypy` و`pytest` و`ruff` تحتاجها **أنت** أثناء التطوير، لكن المستخدم النهائي الذي يُشغّل برنامجك المُنتهي **لا يحتاجها إطلاقًا**. فصلها يُبقي حزمة الإنتاج (Production) خفيفة قدر الإمكان، ويُسرّع عملية النشر (Deployment) لاحقًا (الفصل 26).

### ملف `uv.lock`: الدقة المطلقة

```bash
cat uv.lock | head -20
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

ملف `uv.lock` يُسجّل **بالضبط** رقم الإصدار الدقيق (حتى آخر رقم فرعي) لكل مكتبة وكل تبعياتها الفرعية (Transitive Dependencies)، مع بصمة تشفيرية (Hash) للتحقق من سلامتها. هذا يضمن أن **أي شخص** يستنسخ مشروعك سيحصل على **البيئة المطابقة تمامًا بت لبت**، بغض النظر عن متى أو أين يُشغّل `uv sync`.

> ⚠️ **تحذير:** لا تُعدّل ملف `uv.lock` يدويًا أبدًا! هذا الملف يُدار تلقائيًا بالكامل بواسطة `uv`. يجب إضافته إلى Git (بعكس `.venv/` التي يجب تجاهلها، كما سنرى قريبًا) حتى يحصل كل من يعمل على المشروع على نفس البيئة بالضبط.

### 🔄 مقارنة: `pyproject.toml` مقابل `uv.lock`

| الملف | الوظيفة | مرونة الإصدار |
|---|---|---|
| `pyproject.toml` | يُعرّف **النطاق** المقبول (`requests>=2.31`) | مرن |
| `uv.lock` | يُثبّت **الإصدار الدقيق الفعلي** المُستخدم (`requests==2.31.4`) | صارم تمامًا |

### استنساخ مشروع موجود وتثبيت كل تبعياته دفعة واحدة

عندما تحصل على مشروع (من زميل، أو من Git)، كل ما تحتاجه:

```bash
uv sync
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

هذا الأمر يقرأ `uv.lock` (أو `pyproject.toml` إن لم يوجد lock بعد) وينشئ `.venv/` ويُثبّت **بالضبط** كل ما هو مطلوب، بنفس الإصدارات تمامًا.

### تشغيل الأوامر داخل البيئة: `uv run`

لاحظ أننا استخدمنا `uv run` طوال الكتاب:

```bash
uv run main.py
uv run mypy .
uv run pytest
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

`uv run` تضمن تنفيذ الأمر **داخل** بيئة المشروع الافتراضية مباشرة، دون الحاجة لتفعيلها يدويًا أولًا (عبر `source .venv/bin/activate` التقليدية). هذا أبسط وأقل عرضة للأخطاء (مثل نسيان التفعيل والعمل بالخطأ على بيئة Python عامة أو بيئة مشروع آخر).

> 🔄 **مقارنة: الطريقة التقليدية مقابل uv**
> ```bash
> # الطريقة التقليدية (venv + pip)
> python -m venv .venv
> source .venv/bin/activate        # لينكس/ماك
> .venv\Scripts\activate           # Windows
> pip install requests
> python main.py
> deactivate
>
> # الطريقة الحديثة (uv) - أبسط وأسرع بكثير
> uv add requests
> uv run main.py
> ```

### إدارة إصدارات Python نفسها عبر uv

من الميزات القوية لـ `uv` أنها تستطيع تثبيت وإدارة **إصدارات Python ذاتها**، دون الحاجة لتثبيتها يدويًا من الموقع الرسمي:

```bash
# عرض الإصدارات المتاحة للتثبيت
uv python list

# تثبيت إصدار معيّن
uv python install 3.14

# تحديد إصدار Python لمشروع معيّن
uv python pin 3.14
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

هذا الأمر الأخير يُنشئ/يُحدّث ملف `.python-version` بالقيمة `3.14`، فيضمن أن كل من يعمل على هذا المشروع (بما فيك) يستخدم بالضبط هذا الإصدار.

### تشغيل سكريبتات مستقلة بتبعيات مضمّنة

ميزة فريدة من uv: يمكنك كتابة سكريبت Python **واحد** يُعرّف تبعياته الخاصة بداخله مباشرة، دون الحاجة لمشروع كامل:

In [ ]:
# /// script
# dependencies = [
#   "requests",
# ]
# ///

import requests

response = requests.get("https://api.github.com")
print(response.status_code)

```bash
uv run script.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

`uv` تقرأ التعليق الخاص في الأعلى، تُنشئ بيئة مؤقتة بهذه التبعية بالضبط، وتُشغّل السكريبت فورًا — مثالي لسكريبتات سريعة أو أمثلة مستقلة لا تستحق إنشاء مشروع كامل من أجلها.

### جدول مرجعي شامل لأوامر uv

| الأمر | الوظيفة |
|---|---|
| `uv init project_name` | إنشاء مشروع جديد |
| `uv add package` | إضافة تبعية |
| `uv add --dev package` | إضافة تبعية تطوير فقط |
| `uv remove package` | حذف تبعية |
| `uv sync` | تثبيت كل التبعيات المسجّلة بدقة |
| `uv run command` | تشغيل أمر داخل بيئة المشروع |
| `uv python install 3.14` | تثبيت إصدار Python معيّن |
| `uv python pin 3.14` | تحديد إصدار Python للمشروع |
| `uv lock` | تحديث ملف `uv.lock` يدويًا |
| `uv venv` | إنشاء بيئة افتراضية يدويًا (نادرًا ما نحتاجها مباشرة) |

## 18.3 الجزء الثاني: هيكلة مشروع احترافية

### هيكل "src layout" القياسي

للمشاريع الصغيرة (سكريبت واحد أو اثنين)، وضع كل شيء في المجلد الرئيسي مقبول. لكن للمشاريع الحقيقية، الهيكل القياسي الموصى به عالميًا هو **src layout**:

```
my_project/
├── .venv/
├── .gitignore
├── .python-version
├── pyproject.toml
├── uv.lock
├── README.md
├── src/
│   └── my_project/
│       ├── __init__.py
│       ├── main.py
│       ├── models.py
│       ├── utils.py
│       └── services/
│           ├── __init__.py
│           └── user_service.py
└── tests/
    ├── __init__.py
    ├── test_models.py
    └── test_user_service.py
```

### لماذا `src/` تحديدًا؟

> 💡 **نصيحة الخبير:** وضع الكود داخل `src/my_project/` بدلًا من مباشرة في المجلد الرئيسي يمنع مشكلة شائعة: استيراد غير مقصود للكود «من المجلد الحالي» بدلًا من «النسخة المثبتة فعليًا» للحزمة، مما قد يُخفي أخطاءً حقيقية أثناء التطوير تظهر فقط لاحقًا عند التثبيت الفعلي للمستخدمين. هذا الهيكل أصبح المعيار شبه الرسمي المتفق عليه في مجتمع Python الحديث.

### ملف `__init__.py`: ما وظيفته؟

```python
# src/my_project/__init__.py
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

هذا الملف (قد يكون فارغًا تمامًا) يُخبر Python أن هذا المجلد هو **حزمة (Package)** حقيقية يمكن استيرادها، وليس مجرد مجلد عادي. يمكن أيضًا استخدامه لتصدير عناصر معيّنة بشكل مباشر:

```python
# src/my_project/__init__.py
from my_project.models import User
from my_project.utils import format_date

__version__ = "0.1.0"
```

🧾 **للقراءة فقط** — الوحدة `my_project` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

هذا يسمح للمستخدمين باستيراد `from my_project import User` مباشرة، بدلًا من `from my_project.models import User` الأطول.

### ملف `README.md`: واجهة مشروعك الأولى

كل مشروع احترافي يبدأ بملف `README.md` يشرح:

```markdown
# اسم المشروع

وصف موجز وواضح لما يفعله المشروع.

## التثبيت

\`\`\`bash
uv sync
\`\`\`

## الاستخدام

\`\`\`bash
uv run main.py
\`\`\`

## المتطلبات

- Python 3.14+

## الترخيص

MIT
```

🧾 **للقراءة فقط** — محتوى ملف `markdown`.

> 💡 **نصيحة الخبير:** يُكتب `README.md` بصيغة **Markdown** — لغة تنسيق نصوص بسيطة جدًا (تعرفت عليها ضمنًا في هذا الكتاب نفسه، فكل العناوين والقوائم والصناديق التي تراها مكتوبة بـ Markdown!). `#` للعناوين، ` ```code``` ` للأكواد، `-` للقوائم. تُعرضها منصات مثل GitHub تلقائيًا بتنسيق جميل مقروء.

## 18.4 الجزء الثالث: أساسيات Git

### ما هو Git ولماذا هو لا غنى عنه؟

**Git** هو **نظام التحكم بالإصدارات (Version Control System)** الأكثر انتشارًا في العالم. تخيّله كـ«آلة زمن» لمشروعك: يحفظ كل «لقطة» (Snapshot) من تاريخ تطور كودك، فيمكنك:
- الرجوع لأي نسخة سابقة في أي وقت.
- معرفة من غيّر أي سطر، ومتى، ولماذا.
- العمل على ميزات جديدة بشكل منعزل (Branches) دون المساس بالكود المستقر.
- التعاون مع فريق كامل على نفس الكود دون الكتابة فوق بعضهم البعض.

> 🔄 **مقارنة: Git مقابل «حفظ نسخ يدويًا» (`project_v1`, `project_v2_final`, `project_final_FINAL`)**
> كثير من المبتدئين يحفظون نسخًا متعددة من مجلدهم يدويًا كـ «نسخة احتياطية». هذا الأسلوب كارثي: لا يعرف أحد ما الفرق الدقيق بين النسخ، يستهلك مساحة تخزين هائلة (نسخ كاملة متكررة)، ولا يدعم التعاون الجماعي إطلاقًا. Git يحل كل هذه المشاكل بأناقة: يحفظ فقط **الفروقات (Diffs)** بين كل لقطة والتي تليها، مع رسائل وصفية واضحة لكل تغيير.

### تثبيت Git والتهيئة الأولى

```bash
# تحقق من وجود Git مسبقًا
git --version

# التهيئة لمرة واحدة فقط على جهازك (يُستخدم لتوقيع كل "اللقطات" باسمك)
git config --global user.name "اسمك"
git config --global user.email "بريدك@example.com"
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### دورة حياة Git الأساسية: المفاهيم الجوهرية

Git يعمل بثلاث «مناطق» رئيسية يجب فهمها جيدًا:

```
منطقة العمل          منطقة التجهيز         المستودع (التاريخ)
(Working Directory) → (Staging Area)      → (Repository)
   [تُعدّل ملفاتك]      [git add]              [git commit]
```

1. **Working Directory**: الملفات كما هي الآن على قرصك، قد تحتوي تعديلات غير محفوظة في Git بعد.
2. **Staging Area**: منطقة «تجهيز» مؤقتة — تُحدد فيها **أي** تعديلات تريد تضمينها في اللقطة القادمة.
3. **Repository**: التاريخ الدائم المحفوظ فعليًا — كل «لقطة» مُسجَّلة هنا للأبد (أو حتى تُحذف عمدًا).

### إنشاء أول مستودع Git

```bash
cd my_project
git init
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

هذا الأمر يُنشئ مجلدًا مخفيًا `.git/` يحتوي كل تاريخ المشروع (لا تلمسه يدويًا أبدًا!).

### ملف `.gitignore`: ما الذي لا يجب تتبعه؟

ليس كل شيء في مشروعك يجب أن يُحفظ في Git! الملفات التي تُنشأ تلقائيًا (كالبيئة الافتراضية، أو ملفات الكاش)، أو الملفات الحساسة (كلمات مرور، مفاتيح API)، يجب **استثناؤها** تمامًا.

```gitignore
# البيئة الافتراضية - ضخمة ويمكن إعادة إنشائها بـ uv sync
.venv/

# ملفات بايثون المُجمَّعة
__pycache__/
*.pyc

# ملفات بيئة خاصة (أسرار، مفاتيح API)
.env

# ملفات نظام التشغيل
.DS_Store
Thumbs.db

# ملفات المحرر
.vscode/
*.swp

# ملفات الاختبار والتغطية
.pytest_cache/
.mypy_cache/
htmlcov/
```

🧾 **للقراءة فقط** — محتوى ملف `gitignore`.

> ⚠️ **تحذير أمني بالغ الأهمية:** لا تُضِف أبدًا ملفات تحتوي كلمات مرور أو مفاتيح API سرية إلى Git! إن حدث ذلك بالخطأ، فحتى لو حذفتها لاحقًا، **تبقى في تاريخ Git إلى الأبد** (ما لم تُعِد كتابة التاريخ بالكامل بعملية معقّدة). القاعدة الذهبية: ضع كل الأسرار في ملف `.env` منفصل، وأضفه فورًا إلى `.gitignore` **قبل** أن تكتب فيه أي سر.

> 💡 **نصيحة الخبير:** `uv init` تُنشئ ملف `.gitignore` مبدئي مناسب تلقائيًا عند إنشاء مشروع جديد، لكن من الجيد معرفة محتواه وتعديله حسب احتياجات مشروعك.

### الدورة الأساسية: `add`, `commit`, `status`, `log`

#### `git status`: ماذا تغيّر؟

```bash
git status
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
On branch main
No commits yet
Untracked files:
  (use "git add <file>..." to include in what will be committed)
        main.py
        pyproject.toml
```

يخبرك بالضبط حالة كل ملف: جديد (Untracked)، مُعدَّل (Modified)، أو مُجهَّز (Staged).

#### `git add`: تجهيز التغييرات

```bash
git add main.py          # إضافة ملف واحد محدد
git add .                # إضافة كل الملفات المُعدَّلة/الجديدة في المجلد الحالي وما تحته
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

#### `git commit`: حفظ اللقطة فعليًا

```bash
git commit -m "إضافة الدالة الرئيسية للبرنامج"
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

`-m` تعني «رسالة» (message) — وصف **واضح ومختصر** لما تغيّر ولماذا.

> 💡 **نصيحة الخبير - كتابة رسائل commit جيدة:**
> - ❌ سيئة: `"تعديلات"`, `"fix"`, `"تحديث"`.
> - ✅ جيدة: `"إصلاح خطأ القسمة على صفر في دالة calculate_average"`, `"إضافة ميزة تصدير CSV"`.
> رسالة الـ commit الجيدة تشرح **ماذا** تغيّر و**لماذا**، بحيث يفهمها أي شخص (بما فيك أنت بعد 6 أشهر) دون الحاجة لقراءة الكود بالكامل.

#### `git log`: استعراض التاريخ

```bash
git log
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
commit a1b2c3d4e5f6... (HEAD -> main)
Author: اسمك <بريدك@example.com>
Date:   Mon Jun 10 14:30:00 2025

    إضافة الدالة الرئيسية للبرنامج
```

```bash
git log --oneline    # عرض مختصر، سطر واحد لكل commit - مفيد جدًا للاستعراض السريع
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### دورة عمل يومية نموذجية

```bash
# 1. تحقق من حالة الملفات
git status

# 2. عدّل أكوادك بأي محرر (VS Code مثلاً)
# ... تعديلات ...

# 3. راجع ما تغيّر بالتفصيل (اختياري لكن مفيد)
git diff

# 4. جهّز التغييرات التي أنهيتها منطقيًا
git add src/my_project/utils.py

# 5. احفظ اللقطة برسالة واضحة
git commit -m "إضافة دالة format_currency في utils"

# كرر الدورة...
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

> 💡 **نصيحة الخبير:** اجعل كل commit يمثّل **تغييرًا منطقيًا واحدًا ومتماسكًا** (ميزة واحدة، أو إصلاح خطأ واحد)، بدلًا من تجميع عشرات التغييرات غير المترابطة في commit ضخم واحد. هذا يجعل تاريخ مشروعك أوضح بكثير، ويُسهّل كثيرًا تحديد «أي commit تسبب بهذا الخطأ» لاحقًا إن ظهرت مشكلة.

### الفروع (Branches): العمل المتوازي الآمن

تخيّل أنك تعمل على ميزة جديدة كبيرة، لكن الكود الحالي (`main`) يعمل بشكل مستقر ويُستخدم فعليًا. تريد تجربة أفكار جديدة **دون المخاطرة** بكسر الكود المستقر. هنا يأتي مفهوم **الفروع (Branches)**.

```
main:      A---B---C---------F---G
                    \        /
feature:            D---E---'
```

كل فرع هو «خط تطور مستقل»، يمكن لاحقًا **دمجه (Merge)** مع الفرع الرئيسي بمجرد التأكد من أنه يعمل بشكل صحيح.

```bash
# إنشاء فرع جديد والانتقال إليه مباشرة
git checkout -b feature/user-authentication

# أو بالصيغة الأحدث
git switch -c feature/user-authentication

# ... تعمل، تُعدّل، تعمل commits عادية هنا ...
git add .
git commit -m "إضافة نظام تسجيل الدخول"

# الرجوع للفرع الرئيسي
git switch main

# دمج الفرع الجديد بعد التأكد من جاهزيته
git merge feature/user-authentication

# حذف الفرع بعد دمجه (لم تعد هناك حاجة إليه)
git branch -d feature/user-authentication
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### جدول: أوامر الفروع الأساسية

| الأمر | الوظيفة |
|---|---|
| `git branch` | عرض كل الفروع الموجودة |
| `git switch -c name` | إنشاء فرع جديد والانتقال إليه |
| `git switch name` | الانتقال إلى فرع موجود |
| `git merge name` | دمج فرع معيّن في الفرع الحالي |
| `git branch -d name` | حذف فرع (بعد دمجه بأمان) |

> ⚠️ **تحذير شائع:** تعارضات الدمج (**Merge Conflicts**) تحدث عندما يُعدّل فرعان **نفس السطر بالضبط** بطرق مختلفة. Git لا يعرف أيهما «الصحيح»، فيطلب منك **حسم التعارض يدويًا**. لا داعي للخوف من هذا — هو جزء طبيعي من العمل الجماعي، وVS Code (بإضافة GitLens مثلاً) يُسهّل كثيرًا رؤية وحل هذه التعارضات بصريًا.

### العمل مع مستودعات بعيدة: GitHub

كل ما تعلمناه حتى الآن يعمل **محليًا على جهازك فقط**. للتعاون مع الآخرين (أو لحفظ نسخة احتياطية سحابية)، نحتاج **مستودعًا بعيدًا (Remote Repository)**، وأشهر منصة لاستضافته هي **GitHub**.

### ربط مشروعك المحلي بمستودع GitHub

بعد إنشاء مستودع فارغ على GitHub.com:

```bash
git remote add origin https://github.com/username/my_project.git
git branch -M main
git push -u origin main
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

- `remote add origin ...`: يُخبر Git بعنوان المستودع البعيد، ويسميه `origin` (اسم اصطلاحي شائع).
- `push -u origin main`: يرفع كل الـ commits المحلية إلى GitHub، ويربط الفرع المحلي `main` بنظيره البعيد (`-u` تعني «تذكر هذا الربط للمستقبل»).

### العمليات اليومية مع مستودع بعيد

```bash
git push                  # رفع commits جديدة محلية إلى GitHub
git pull                  # تنزيل آخر تحديثات من GitHub ودمجها محليًا
git clone <url>           # نسخ مستودع كامل من GitHub إلى جهازك لأول مرة
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

> 💡 **نصيحة الخبير:** `git clone` هي أول أمر ستستخدمه عند الانضمام لأي مشروع جماعي موجود مسبقًا على GitHub. بعدها مباشرة، نفّذ `uv sync` لتثبيت كل التبعيات المطلوبة بدقة تامة، وستكون جاهزًا للعمل خلال دقائق.

### 🔄 مقارنة: Git مقابل GitHub

| | Git | GitHub |
|---|---|---|
| ما هو؟ | أداة برمجية (نظام تحكم بالإصدارات) | منصة سحابية لاستضافة مستودعات Git |
| أين يعمل؟ | محليًا على جهازك | على الإنترنت |
| هل ضروري؟ | نعم، لا بديل عنه عمليًا | اختياري (بدائل: GitLab, Bitbucket) لكنه الأشهر |

### جدول مرجعي شامل لأوامر Git

| الأمر | الوظيفة |
|---|---|
| `git init` | إنشاء مستودع جديد |
| `git status` | عرض حالة الملفات الحالية |
| `git add <file>` / `git add .` | تجهيز التغييرات |
| `git commit -m "..."` | حفظ لقطة بالتغييرات المُجهَّزة |
| `git log` / `git log --oneline` | عرض تاريخ الـ commits |
| `git diff` | عرض الفروقات التفصيلية غير المُجهَّزة بعد |
| `git branch` | عرض الفروع |
| `git switch -c <name>` | إنشاء فرع جديد والانتقال إليه |
| `git switch <name>` | التبديل بين الفروع |
| `git merge <name>` | دمج فرع في الفرع الحالي |
| `git remote add origin <url>` | ربط مستودع بعيد |
| `git push` | رفع التغييرات للمستودع البعيد |
| `git pull` | تنزيل ودمج تحديثات المستودع البعيد |
| `git clone <url>` | نسخ مستودع بعيد كاملاً محليًا |

## 18.5 أفضل الممارسات الشاملة

### Commit مبكرًا، Commit كثيرًا

لا تنتظر حتى تنتهي «من كل شيء» لعمل commit واحد ضخم. كل تغيير منطقي صغير يستحق commit خاصًا به.

### لا تضع الأسرار في Git أبدًا

```bash
# .env (لا يُضاف لـ Git أبدًا)
API_KEY=sk-1234567890abcdef
DATABASE_PASSWORD=super_secret_password
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```gitignore
# .gitignore
.env
```

🧾 **للقراءة فقط** — محتوى ملف `gitignore`.

### اكتب رسائل commit بصيغة الأمر (اصطلاح شائع)

```bash
# ✅ اصطلاح شائع عالميًا
git commit -m "Add user authentication"
git commit -m "Fix division by zero bug"

# بدلًا من
git commit -m "Added user authentication"  # صيغة الماضي
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

(هذا اصطلاح بالإنجليزية شائع عالميًا في المشاريع مفتوحة المصدر، لكن إن كان مشروعك محليًا بالعربية بالكامل، استخدم العربية بوضوح واتساق).

### لا تُضف `.venv/` أبدًا إلى Git

البيئة الافتراضية ضخمة ويمكن إعادة إنشائها بالكامل من `pyproject.toml` و `uv.lock` في ثوانٍ عبر `uv sync`. إضافتها لـ Git يُضخّم المستودع دون أي فائدة.

### استخدم فروعًا منفصلة للميزات الكبيرة

تجنّب العمل مباشرة على `main` للميزات الكبيرة أو التجريبية — اعمل في فرع منفصل، وادمجه فقط بعد التأكد من استقراره.

> ⚠️ **خطأ شائع:** نسيان `git add` قبل `git commit`! إن نسيت، ستظهر رسالة "nothing to commit" أو سيُحفظ commit فارغ. تحقق دائمًا بـ `git status` قبل `git commit` للتأكد من أن التغييرات المطلوبة فعلاً «مُجهَّزة».

## 18.6 مشروع تطبيقي: تجهيز مشروع كامل من الصفر

لنطبّق كل ما تعلمناه في سيناريو متكامل: إنشاء مشروع «محول عملات» بسيط، بهيكلية احترافية كاملة، وتاريخ Git نظيف، وجاهز للرفع على GitHub.

<details>
<summary>✅ خطوات الحل الكاملة</summary>

### الخطوة 1: إنشاء المشروع

```bash
uv init currency_converter
cd currency_converter
uv python pin 3.14
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخطوة 2: إعادة الهيكلة إلى src layout

```bash
mkdir -p src/currency_converter tests
touch src/currency_converter/__init__.py
touch tests/__init__.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخطوة 3: كتابة الكود الأساسي

```python
# src/currency_converter/__init__.py
from currency_converter.converter import convert

__version__ = "0.1.0"
__all__ = ["convert"]
```

🧾 **للقراءة فقط** — الوحدة `currency_converter` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

In [ ]:
# src/currency_converter/converter.py
"""وحدة تحويل العملات."""
from __future__ import annotations

EXCHANGE_RATES: dict[str, float] = {
    "USD": 1.0,
    "EUR": 0.92,
    "SAR": 3.75,
    "EGP": 48.5,
}

class UnsupportedCurrencyError(Exception):
    """تُرفع عندما تكون العملة المطلوبة غير مدعومة."""

def convert(amount: float, from_currency: str, to_currency: str) -> float:
    """
    يحوّل مبلغًا من عملة إلى أخرى بناءً على أسعار الصرف الثابتة.

    Args:
        amount: المبلغ المراد تحويله.
        from_currency: رمز العملة الأصلية (مثل "USD").
        to_currency: رمز العملة الهدف (مثل "SAR").

    Returns:
        المبلغ المحوّل بالعملة الهدف.

    Raises:
        UnsupportedCurrencyError: إن كانت إحدى العملتين غير مدعومة.
    """
    if from_currency not in EXCHANGE_RATES:
        raise UnsupportedCurrencyError(f"العملة غير مدعومة: {from_currency}")
    if to_currency not in EXCHANGE_RATES:
        raise UnsupportedCurrencyError(f"العملة غير مدعومة: {to_currency}")

    amount_in_usd = amount / EXCHANGE_RATES[from_currency]
    return round(amount_in_usd * EXCHANGE_RATES[to_currency], 2)

```python
# src/currency_converter/main.py
"""نقطة الدخول الرئيسية للبرنامج."""
from currency_converter.converter import convert, UnsupportedCurrencyError

def main() -> None:
    try:
        result = convert(100, "USD", "SAR")
        print(f"100 USD = {result} SAR")
    except UnsupportedCurrencyError as e:
        print(f"خطأ: {e}")

if __name__ == "__main__":
    main()
```

🧾 **للقراءة فقط** — الوحدة `currency_converter` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

### الخطوة 4: ملف `pyproject.toml` المُحدَّث

```toml
[project]
name = "currency-converter"
version = "0.1.0"
description = "محول عملات بسيط"
readme = "README.md"
requires-python = ">=3.14"
dependencies = []

[dependency-groups]
dev = [
    "mypy>=1.8.0",
    "pytest>=8.0.0",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.mypy]
strict = true
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

### الخطوة 5: كتابة `README.md`

```markdown
# محول العملات

أداة بسيطة لتحويل المبالغ بين عملات مختلفة بأسعار صرف ثابتة.

## التثبيت

\`\`\`bash
uv sync
\`\`\`

## الاستخدام

\`\`\`bash
uv run python -m currency_converter.main
\`\`\`

## الاختبار

\`\`\`bash
uv run pytest
uv run mypy .
\`\`\`
```

🧾 **للقراءة فقط** — محتوى ملف `markdown`.

### الخطوة 6: تهيئة Git

```bash
git init
git add .
git commit -m "الإعداد الأولي للمشروع: هيكلية وتحويل عملات أساسي"
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخطوة 7: التحقق

```bash
uv run mypy . --strict
# Success: no issues found

uv run python -c "from currency_converter.converter import convert; print(convert(100, 'USD', 'SAR'))"
# 375.0

git log --oneline
# a1b2c3d الإعداد الأولي للمشروع: هيكلية وتحويل عملات أساسي
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخطوة 8 (اختياري): الرفع على GitHub

```bash
git remote add origin https://github.com/username/currency_converter.git
git branch -M main
git push -u origin main
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخطوة 9: إضافة ميزة جديدة في فرع منفصل

```bash
git switch -c feature/add-more-currencies
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

In [ ]:
# تعديل converter.py - إضافة عملات جديدة
EXCHANGE_RATES: dict[str, float] = {
    "USD": 1.0,
    "EUR": 0.92,
    "SAR": 3.75,
    "EGP": 48.5,
    "AED": 3.67,   # جديد
    "KWD": 0.31,   # جديد
}

```bash
git add .
git commit -m "إضافة دعم عملتي AED و KWD"
git switch main
git merge feature/add-more-currencies
git branch -d feature/add-more-currencies
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

</details>

## 18.7 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1:** أنشئ مشروعًا جديدًا باسم `my_first_project` باستخدام `uv init`، ثبّت فيه مكتبة `requests`، وتحقق من ظهورها في `pyproject.toml`.

**تمرين 2:** في مشروعك من التمرين السابق، نفّذ `git init`، ثم اعمل أول commit برسالة مناسبة.

**تمرين 3:** ما الفرق بين `git add` و `git commit`؟ اشرح بجملتين.

<details>
<summary>✅ الحل النموذجي</summary>

```bash
# تمرين 1
uv init my_first_project
cd my_first_project
uv add requests
cat pyproject.toml  # تحقق من وجود requests ضمن dependencies

# تمرين 2
git init
git add .
git commit -m "الإعداد الأولي للمشروع"

# تمرين 3
# git add: يُجهّز (Stage) التغييرات، أي يُحدد أيها سيدخل في اللقطة القادمة، دون حفظها نهائيًا بعد.
# git commit: يحفظ فعليًا كل التغييرات المُجهَّزة كلقطة دائمة في تاريخ المشروع، برسالة وصفية.
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

</details>


### 🟡 مستوى متوسط

**تمرين 4:** أعد هيكلة مشروع بسيط (يمكنك استخدام مشروع «مدير جهات الاتصال» من الفصل 16) إلى `src layout` قياسي، مع `tests/` فارغة جاهزة للفصل 25.

**تمرين 5:** أنشئ فرعًا جديدًا باسم `feature/export-pdf`، اعمل تعديلًا وهميًا بسيطًا (مثل إضافة دالة فارغة)، اعمل commit، ثم ادمجه في `main`.

**تمرين 6:** أنشئ ملف `.gitignore` مناسب لمشروع Python كامل، يتضمن: البيئة الافتراضية، ملفات الكاش، ملفات `.env`، وملفات نظام التشغيل.

<details>
<summary>✅ الحل النموذجي</summary>

```bash
# تمرين 4
mkdir -p src/contact_manager tests
touch src/contact_manager/__init__.py tests/__init__.py
# نقل contact_manager.py الأصلي إلى src/contact_manager/manager.py
# وتحديث الاستيرادات بما يناسب الهيكلية الجديدة

# تمرين 5
git switch -c feature/export-pdf
# إضافة دالة export_to_pdf فارغة في الكود
git add .
git commit -m "إضافة دالة export_to_pdf (هيكل أولي)"
git switch main
git merge feature/export-pdf
git branch -d feature/export-pdf

# تمرين 6
cat > .gitignore << 'EOF'
.venv/
__pycache__/
*.pyc
.env
.DS_Store
Thumbs.db
.vscode/
.pytest_cache/
.mypy_cache/
htmlcov/
EOF
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

</details>


### 🔴 تحدي

**تمرين 7:** محاكِ سيناريو عمل جماعي كامل:
1. أنشئ مشروعًا جديدًا، واعمل commit أولي على `main`.
2. أنشئ فرعين منفصلين: `feature/login` و `feature/signup`، وفي كل منهما أضف دالة بسيطة مختلفة في نفس الملف (لنفس السطر تقريبًا) لإجبار حدوث تعارض عمدًا.
3. ادمج `feature/login` في `main` بنجاح.
4. حاول دمج `feature/signup` في `main` وشاهد رسالة التعارض (Merge Conflict).
5. افتح الملف المتعارض، لاحظ العلامات الخاصة (`<<<<<<<`, `=======`, `>>>>>>>`)، احسم التعارض يدويًا بدمج كلا التغييرين منطقيًا، ثم أكمل الدمج بـ `git add` و `git commit`.

<details>
<summary>✅ إرشادات الحل</summary>

```bash
# 1. الإعداد
uv init conflict_demo
cd conflict_demo
echo "def main():
    print('البرنامج الرئيسي')
" > app.py
git init
git add .
git commit -m "الإعداد الأولي"

# 2. فرع login
git switch -c feature/login
echo "def main():
    print('ميزة تسجيل الدخول')
" > app.py
git add .
git commit -m "إضافة ميزة تسجيل الدخول"

git switch main

# فرع signup (من main، لإحداث تعارض حقيقي على نفس الأسطر)
git switch -c feature/signup
echo "def main():
    print('ميزة إنشاء حساب جديد')
" > app.py
git add .
git commit -m "إضافة ميزة إنشاء حساب"

git switch main

# 3. دمج أول فرع - ينجح بلا مشاكل
git merge feature/login

# 4. محاولة دمج الفرع الثاني - تعارض!
git merge feature/signup
# Auto-merging app.py
# CONFLICT (content): Merge conflict in app.py

# 5. فتح app.py، سيظهر شيء كهذا:
# <<<<<<< HEAD
# def main():
#     print('ميزة تسجيل الدخول')
# =======
# def main():
#     print('ميزة إنشاء حساب جديد')
# >>>>>>> feature/signup

# الحسم اليدوي - دمج كلا الميزتين منطقيًا:
cat > app.py << 'EOF'
def main():
    print('ميزة تسجيل الدخول')
    print('ميزة إنشاء حساب جديد')
EOF

git add app.py
git commit -m "حسم التعارض: دمج ميزتي تسجيل الدخول وإنشاء الحساب"
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

هذا التمرين يُحاكي بدقة الموقف الذي سيواجهه كل مبرمج يعمل ضمن فريق عاجلًا أم آجلًا — وفهمه مبكرًا يُزيل الخوف منه تمامًا.
</details>

## 18.8 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: لماذا نحتاج بيئة افتراضية منفصلة لكل مشروع؟</summary>

لتجنّب تعارض إصدارات المكتبات بين المشاريع المختلفة. كل مشروع قد يحتاج إصدارًا مختلفًا من نفس المكتبة، والتثبيت العام (Global) على الجهاز يسمح بإصدار واحد فقط في كل مرة، مما يُسبب تعارضات حتمية عند العمل على عدة مشاريع.
</details>

<details>
<summary>❓ السؤال 2: ما الفرق بين `pyproject.toml` و `uv.lock`؟</summary>

`pyproject.toml` يُعرّف **النطاق** المقبول لكل تبعية (مثل `requests>=2.31`)، بينما `uv.lock` يُسجّل **الإصدار الدقيق الفعلي** المُستخدم فعليًا لكل تبعية وتبعياتها الفرعية، لضمان أن كل من يُشغّل `uv sync` يحصل على نفس البيئة تمامًا.
</details>

<details>
<summary>❓ السؤال 3: ما وظيفة ملف `.gitignore`؟</summary>

يُخبر Git بالملفات والمجلدات التي يجب **تجاهلها تمامًا** وعدم تتبعها، مثل البيئة الافتراضية (`.venv/`)، ملفات الكاش المؤقتة، والملفات الحساسة كـ `.env` التي تحتوي أسرارًا لا يجب رفعها لمستودع مشترك.
</details>

<details>
<summary>❓ السؤال 4: اشرح الفرق بين `git add`, `git commit`, و `git push`.</summary>

`git add` يُجهّز التغييرات (ينقلها من Working Directory إلى Staging Area). `git commit` يحفظها فعليًا كلقطة دائمة في تاريخ المستودع المحلي. `git push` يرفع هذه اللقطات المحفوظة محليًا إلى مستودع بعيد (مثل GitHub) ليراها الآخرون.
</details>

<details>
<summary>❓ السؤال 5: لماذا نستخدم الفروع (Branches) في Git؟</summary>

للعمل على ميزات جديدة أو تجريبية بشكل منعزل تمامًا عن الفرع الرئيسي المستقر (`main`)، مما يسمح بالتطوير والتجربة دون المخاطرة بكسر الكود الذي يعمل بالفعل. بعد التأكد من جاهزية الميزة، تُدمج في الفرع الرئيسي بأمان.
</details>

<details>
<summary>❓ السؤال 6: ماذا يحدث عند تعارض الدمج (Merge Conflict)، وكيف يُحل؟</summary>

يحدث عندما يُعدّل فرعان مختلفان **نفس الأسطر بالضبط** بطرق متضاربة، فلا يستطيع Git تحديد النسخة الصحيحة تلقائيًا. يُحل يدويًا بفتح الملف المتعارض، ومراجعة العلامات الخاصة (`<<<<<<<`, `=======`, `>>>>>>>`) التي تُظهر كلا النسختين، واختيار أو دمج المحتوى الصحيح يدويًا، ثم `git add` و `git commit` لإكمال عملية الدمج.
</details>

## 18.9 ملخص الفصل (Cheat Sheet)

```bash
# ===== uv: إدارة المشروع =====
uv init project_name          # مشروع جديد
uv add package                # إضافة تبعية
uv add --dev package          # تبعية تطوير فقط
uv remove package             # حذف تبعية
uv sync                       # تثبيت كل التبعيات بدقة
uv run command                # تشغيل أمر داخل بيئة المشروع
uv python pin 3.14            # تحديد إصدار Python

# ===== هيكلية src layout =====
# my_project/
# ├── src/my_project/__init__.py
# ├── tests/
# ├── pyproject.toml
# ├── uv.lock
# ├── .gitignore
# └── README.md

# ===== Git: الأساسيات =====
git init                       # إنشاء مستودع
git status                     # عرض الحالة
git add .                      # تجهيز كل التغييرات
git commit -m "رسالة واضحة"    # حفظ لقطة
git log --oneline              # عرض التاريخ

# ===== Git: الفروع =====
git switch -c feature/name     # فرع جديد والانتقال إليه
git switch main                # العودة للفرع الرئيسي
git merge feature/name         # دمج فرع
git branch -d feature/name     # حذف فرع بعد دمجه

# ===== Git: العمل البعيد (GitHub) =====
git remote add origin <url>    # ربط مستودع بعيد
git push -u origin main        # رفع أول مرة
git push                       # رفع التحديثات
git pull                       # سحب آخر التحديثات
git clone <url>                # نسخ مستودع كامل
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

| المفهوم | الأداة |
|---|---|
| عزل تبعيات المشروع | `.venv/` عبر `uv` |
| توثيق التبعيات الدقيقة | `uv.lock` |
| هيكلية احترافية | `src/` layout |
| تاريخ التغييرات | Git (`commit`) |
| عمل متوازٍ آمن | Git Branches |
| استضافة سحابية ومشاركة | GitHub |

## 18.10 ماذا بعد؟

تهانينا! لقد أكملت الآن **الجزء الثالث: Python الحديثة والاحترافية** بالكامل. تمتلك الآن كل الأدوات الأساسية: Type Hints للوضوح والأمان، معالجة الملفات وتنسيقات البيانات، البرمجة غير المتزامنة للأداء، وإدارة مشاريع احترافية عبر uv وGit. هذه المهارات مجتمعة تجعلك قادرًا على الانتقال من «كتابة سكريبتات تعليمية» إلى «بناء برامج حقيقية قابلة للمشاركة والصيانة والتطوير الجماعي».

الآن، وقد اكتملت الأساسيات الاحترافية، حان الوقت لأمر مثير: **تطبيق كل ما تعلمناه على مسارات تخصص حقيقية**. في الجزء الرابع من الكتاب، سننتقل عبر خمسة مسارات مختلفة تمامًا، كل واحد يمثل مجال عمل حقيقي يستخدم Python بكثافة في سوق العمل اليوم.

سنبدأ بمسار **تطوير الويب**: سنبني معًا واجهة برمجة تطبيقات (API) كاملة لإدارة المهام، باستخدام إطار العمل الحديث **FastAPI** — الذي يعتمد بشكل أساسي على كل ما تعلمناه للتو (Type Hints، async/await، وهيكلة المشاريع)!

---

### الخطوة التالية

- [الفصل التاسع عشر: مسار تطوير الويب - بناء واجهة API لإدارة المهام (FastAPI)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-19.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الثامن عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*